# Simple Transformer - GPT from Scratch

A character-level GPT built from scratch for learning. Trains on Tiny Shakespeare on a T4 GPU.

Architecture follows karpathy/nanochat:
- Pre-norm (RMSNorm before each sublayer)
- ReLU² activation
- No bias in linear layers
- Untied embedding / lm_head weights

## 1. Setup

In [ ]:
!pip install torch numpy requests

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")

## 2. Config

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = 256       # character-level: 256 possible byte values
    n_embd: int = 64            # embedding dimension
    n_head: int = 4             # number of attention heads
    n_layer: int = 4            # number of transformer blocks
    seq_len: int = 256          # maximum sequence length
    dropout: float = 0.0        # dropout rate (0 = no dropout)

# # Bigger config for T4 GPU:
# @dataclass
# class GPTConfig:
#     vocab_size: int = 256
#     n_embd: int = 256
#     n_head: int = 8
#     n_layer: int = 8
#     seq_len: int = 512
#     dropout: float = 0.0

config = GPTConfig()
print(f"Config: {config}")

## 3. Model

In [ ]:
import math

import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------------------------
# LayerNorm normalizes by subtracting mean and dividing by std.
# RMSNorm is simpler: it only divides by the root-mean-square (no mean subtraction).
# This works just as well in practice and is slightly faster.
#
# Formula: RMSNorm(x) = x / RMS(x) * gamma
#   where RMS(x) = sqrt(mean(x^2) + eps)
#   and gamma is a learnable per-feature scale

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # learnable scale (gamma)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, seq_len, dim)
        # Compute RMS across the last dimension (the embedding dimension)
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        # Normalize and apply learnable scale
        return (x / rms) * self.weight


# ---------------------------------------------------------------------------
# Causal Self-Attention
# ---------------------------------------------------------------------------
# This is the core of the transformer. Here's the intuition:
#
# Each token asks: "What should I pay attention to?"
# - Q (query): "What am I looking for?"
# - K (key):   "What do I contain?"
# - V (value): "What information do I provide?"
#
# Attention = softmax(Q @ K^T / sqrt(d_k)) @ V
#
# The scaling by sqrt(d_k) prevents the dot products from growing too large.
# Without it, softmax would produce near-one-hot distributions (vanishing gradients).
#
# The causal mask ensures each position can only attend to previous positions
# (and itself). This is what makes it autoregressive.
#
# Multi-head: we split Q, K, V into multiple "heads" that attend independently,
# then concatenate the results.

class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0, "n_embd must be divisible by n_head"

        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head  # dimension per head

        # Separate projections for Q, K, V (no bias, following nanochat)
        self.W_q = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.W_k = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.W_v = nn.Linear(config.n_embd, config.n_embd, bias=False)

        # Output projection: combines multi-head outputs back to n_embd
        self.W_o = nn.Linear(config.n_embd, config.n_embd, bias=False)

        self.dropout = nn.Dropout(config.dropout)

        # Precompute the causal mask
        causal_mask = torch.tril(torch.ones(config.seq_len, config.seq_len))
        self.register_buffer("causal_mask", causal_mask.view(1, 1, config.seq_len, config.seq_len))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape  # batch, sequence length, embedding dim

        # Step 1: Project input into Q, K, V
        q = self.W_q(x)  # (B, T, C)
        k = self.W_k(x)  # (B, T, C)
        v = self.W_v(x)  # (B, T, C)

        # Step 2: Reshape into multiple heads
        # (B, T, C) -> (B, T, n_head, head_dim) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Step 3: Compute attention scores
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # Shape: (B, n_head, T, T)

        # Step 4: Apply causal mask
        attn = attn.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))

        # Step 5: Softmax to get attention weights
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        # Step 6: Weighted sum of values
        out = attn @ v  # (B, n_head, T, head_dim)

        # Step 7: Concatenate heads and project back to n_embd
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_o(out)

        return out


# ---------------------------------------------------------------------------
# MLP (Feed-Forward Network)
# ---------------------------------------------------------------------------
# After attention figures out "what to look at", the MLP does the "thinking".
# Expansion factor of 4x is standard.
# ReLU\u00b2 (ReLU squared): relu(x)^2 instead of GELU.

class MLP(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.c_fc(x)               # (B, T, 4*C)
        x = F.relu(x).square()         # ReLU\u00b2 activation
        x = self.c_proj(x)             # (B, T, C)
        x = self.dropout(x)
        return x


# ---------------------------------------------------------------------------
# Transformer Block
# ---------------------------------------------------------------------------
# Pre-norm residual: x = x + sublayer(norm(x))

class Block(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = RMSNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


# ---------------------------------------------------------------------------
# GPT Model
# ---------------------------------------------------------------------------

class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.seq_len, config.n_embd)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Initialize weights
        self.apply(self._init_weights)
        # Scale residual projections by 1/sqrt(2*n_layer)
        for name, p in self.named_parameters():
            if name.endswith("W_o.weight") or name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.seq_len, f"Sequence length {T} exceeds max {self.config.seq_len}"

        pos = torch.arange(T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        print(f"Total parameters: {total:,}")
        print(f"  Token embeddings (wte):    {self.wte.weight.numel():,}")
        print(f"  Position embeddings (wpe): {self.wpe.weight.numel():,}")
        for i, block in enumerate(self.blocks):
            block_params = sum(p.numel() for p in block.parameters())
            print(f"  Block {i}: {block_params:,}")
        print(f"  Final norm:                {sum(p.numel() for p in self.ln_f.parameters()):,}")
        print(f"  LM head:                   {self.lm_head.weight.numel():,}")
        return total

print("Model classes defined: RMSNorm, CausalSelfAttention, MLP, Block, GPT")

## 4. Model Test

In [ ]:
# Instantiate model and verify forward pass
model = GPT(config)
model.count_parameters()

# Test forward pass with random token IDs
idx = torch.randint(0, config.vocab_size, (2, 32))  # batch=2, seq=32
logits, loss = model(idx)
assert logits.shape == (2, 32, config.vocab_size), f"Logit shape wrong: {logits.shape}"
print(f"\nForward pass: idx {idx.shape} -> logits {logits.shape}")

# Test with targets - loss should be ~ln(vocab_size) for random init
targets = torch.randint(0, config.vocab_size, (2, 32))
logits, loss = model(idx, targets)
expected_loss = math.log(config.vocab_size)
print(f"Loss: {loss.item():.4f} (expected ~{expected_loss:.4f} for random init)")
assert abs(loss.item() - expected_loss) < 1.0, "Loss too far from expected"

# Test gradient flow
loss.backward()
grad_norms = {name: p.grad.norm().item() for name, p in model.named_parameters() if p.grad is not None}
assert len(grad_norms) > 0, "No gradients computed"
print(f"Gradient flow: all {len(grad_norms)} parameters have gradients")

# Clean up test model
del model, logits, loss, idx, targets
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\nAll model tests passed!")

## 5. Data

In [ ]:
import os
import requests

# ---------------------------------------------------------------------------
# Character-level tokenizer
# ---------------------------------------------------------------------------
# The simplest tokenizer: each character is a token.
# vocab_size = 256 (one for each byte value).

def encode(text: str) -> list[int]:
    """Convert a string to a list of byte values (0-255)."""
    return list(text.encode("utf-8"))

def decode(tokens: list[int]) -> str:
    """Convert a list of byte values back to a string."""
    return bytes(tokens).decode("utf-8", errors="replace")


# ---------------------------------------------------------------------------
# Dataset download
# ---------------------------------------------------------------------------

DATA_DIR = "/content/data"
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

def download_shakespeare() -> str:
    """Download tiny Shakespeare and return the text."""
    os.makedirs(DATA_DIR, exist_ok=True)
    filepath = os.path.join(DATA_DIR, "shakespeare.txt")

    if not os.path.exists(filepath):
        print(f"Downloading tiny Shakespeare to {filepath}...")
        response = requests.get(SHAKESPEARE_URL)
        response.raise_for_status()
        with open(filepath, "w") as f:
            f.write(response.text)
        print(f"Downloaded {len(response.text):,} characters.")

    with open(filepath, "r") as f:
        return f.read()


# ---------------------------------------------------------------------------
# Batch construction
# ---------------------------------------------------------------------------

def get_batch(data: torch.Tensor, batch_size: int, seq_len: int, device: str = "cpu"):
    """Sample a random batch of (input, target) pairs from the data."""
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[i : i + seq_len] for i in starts])
    y = torch.stack([data[i + 1 : i + 1 + seq_len] for i in starts])
    return x.to(device), y.to(device)


def prepare_data(val_fraction: float = 0.1):
    """Download Shakespeare, encode it, and split into train/val."""
    text = download_shakespeare()
    tokens = encode(text)
    data = torch.tensor(tokens, dtype=torch.long)

    split_idx = int(len(data) * (1 - val_fraction))
    train_data = data[:split_idx]
    val_data = data[split_idx:]

    return train_data, val_data

print("Data functions defined: encode, decode, download_shakespeare, get_batch, prepare_data")

## 6. Data Test

In [ ]:
# Test tokenizer
text = "Hello, World!"
tokens = encode(text)
decoded = decode(tokens)
assert decoded == text, f"Round-trip failed: {decoded!r} != {text!r}"
print(f"Tokenizer: '{text}' -> {tokens} -> '{decoded}'")
print(f"Vocab size: 256 (one per byte)")
print()

# Download and prepare data
train_data, val_data = prepare_data()
print(f"Dataset: {len(train_data) + len(val_data):,} total tokens")
print(f"  Train: {len(train_data):,} tokens")
print(f"  Val:   {len(val_data):,} tokens")
print()

# Test batch construction
x, y = get_batch(train_data, batch_size=4, seq_len=32)
print(f"Batch: x={x.shape}, y={y.shape}")
print(f"  First input:  '{decode(x[0].tolist())}'")
print(f"  First target: '{decode(y[0].tolist())}'")
assert torch.equal(x[0, 1:], y[0, :-1]), "Target should be input shifted right by 1"
print(f"  Shift-by-1 verified")
print()
print("All data tests passed!")

## 7. Training Utilities

In [ ]:
import time

# ---------------------------------------------------------------------------
# Learning rate schedule
# ---------------------------------------------------------------------------
# Two phases:
#   1. Warmup: linearly increase LR from 0 to max over first N steps.
#   2. Cosine decay: smoothly decrease LR from max to min.

def get_lr(step: int, max_lr: float, min_lr: float, warmup_steps: int, total_steps: int) -> float:
    """Compute learning rate for a given step (warmup + cosine decay)."""
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + (max_lr - min_lr) * cosine


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------

@torch.no_grad()
def estimate_loss(model, train_data, val_data, batch_size, seq_len, device, eval_iters=20):
    """Estimate train and val loss by averaging over several batches."""
    model.eval()
    results = {}
    for name, data in [("train", train_data), ("val", val_data)]:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(data, batch_size, seq_len, device)
            _, loss = model(x, y)
            losses.append(loss.item())
        results[name] = sum(losses) / len(losses)
    model.train()
    return results


# ---------------------------------------------------------------------------
# Checkpointing
# ---------------------------------------------------------------------------

def save_checkpoint(model, config, optimizer, step, path):
    """Save model, optimizer, and config to a file."""
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": config,
        "step": step,
    }, path)

def load_checkpoint(path, device="cpu"):
    """Load a checkpoint and return (model, config, step)."""
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    config = checkpoint["config"]
    model = GPT(config).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model, config, checkpoint["step"]

print("Training utilities defined: get_lr, estimate_loss, save_checkpoint, load_checkpoint")

## 8. Training

In [ ]:
# Training hyperparameters
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

max_steps = 500       # increase to 3000+ with a GPU
batch_size = 32
max_lr = 3e-3
min_lr = 3e-4
warmup_steps = 50
eval_interval = 100
save_interval = 250
checkpoint_dir = "/content/checkpoints"

print("=" * 60)
print("TRAINING")
print("=" * 60)
print(f"Device: {device}")
print(f"Config: n_embd={config.n_embd}, n_head={config.n_head}, n_layer={config.n_layer}")
print(f"Steps: {max_steps}, Batch size: {batch_size}, Seq len: {config.seq_len}")
print(f"LR: {max_lr} -> {min_lr} (warmup {warmup_steps} steps)")
print()

# Prepare data and model
train_data, val_data = prepare_data()
model = GPT(config).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")

# AdamW optimizer with weight decay separation
decay_params = [p for n, p in model.named_parameters() if p.dim() >= 2]
nodecay_params = [p for n, p in model.named_parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": 0.1},
    {"params": nodecay_params, "weight_decay": 0.0},
], lr=max_lr, betas=(0.9, 0.95))

print(f"Optimizer groups: {len(decay_params)} decay, {len(nodecay_params)} no-decay")
print()

# Mixed precision training (only on GPU)
use_amp = device in ("cuda", "mps")
scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
amp_dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

os.makedirs(checkpoint_dir, exist_ok=True)
best_val_loss = float("inf")
t0 = time.time()

for step in range(max_steps):
    # Update learning rate
    lr = get_lr(step, max_lr, min_lr, warmup_steps, max_steps)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    # Forward + backward
    x, y = get_batch(train_data, batch_size, config.seq_len, device)

    if use_amp:
        with torch.autocast(device_type=device, dtype=amp_dtype):
            _, loss = model(x, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    optimizer.zero_grad(set_to_none=True)

    # Logging
    if step % 50 == 0:
        elapsed = time.time() - t0
        tokens_per_sec = (step + 1) * batch_size * config.seq_len / elapsed if elapsed > 0 else 0
        print(f"step {step:5d} | loss {loss.item():.4f} | lr {lr:.2e} | {tokens_per_sec:.0f} tok/s")

    # Evaluation
    if step > 0 and step % eval_interval == 0:
        losses = estimate_loss(model, train_data, val_data, batch_size, config.seq_len, device)
        print(f"  >>> eval | train {losses['train']:.4f} | val {losses['val']:.4f}")

        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            save_checkpoint(model, config, optimizer, step, os.path.join(checkpoint_dir, "best.pt"))
            print(f"  >>> new best val loss, saved checkpoint")

    # Periodic save
    if step > 0 and step % save_interval == 0:
        save_checkpoint(model, config, optimizer, step, os.path.join(checkpoint_dir, "latest.pt"))

# Final save
save_checkpoint(model, config, optimizer, max_steps, os.path.join(checkpoint_dir, "final.pt"))
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

## 9. Generation

In [ ]:
# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
# Autoregressive generation: predict one token at a time, append, repeat.
#
# Temperature controls randomness:
#   - temp=0: greedy (always pick most likely) - deterministic
#   - temp=0.8: fairly conservative, mostly coherent
#   - temp=1.0: sample from model's distribution as-is
#
# Top-k restricts sampling to only the k most likely tokens.

@torch.no_grad()
def generate(model, prompt_tokens: list[int], max_new_tokens: int = 200,
             temperature: float = 0.8, top_k: int = 50, device: str = "cpu") -> list[int]:
    """Generate tokens autoregressively."""
    model.eval()
    seq_len = model.config.seq_len
    tokens = torch.tensor([prompt_tokens], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx = tokens[:, -seq_len:]
        logits, _ = model(idx)
        logits = logits[:, -1, :]  # (1, vocab_size)

        if temperature == 0:
            next_token = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k > 0:
                top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < top_values[:, -1:]] = float("-inf")
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        tokens = torch.cat([tokens, next_token], dim=1)

    return tokens[0].tolist()


# Generate with different prompts and temperatures
prompts = [
    ("ROMEO:", 0.8),
    ("To be or not to be", 0.5),
    ("The king", 1.0),
]

for prompt_text, temp in prompts:
    print("=" * 60)
    print(f"Prompt: '{prompt_text}' | Temperature: {temp}")
    print("-" * 60)

    prompt_tokens = encode(prompt_text) if prompt_text else [0]
    output_tokens = generate(
        model,
        prompt_tokens,
        max_new_tokens=500,
        temperature=temp,
        top_k=50,
        device=device,
    )
    print(decode(output_tokens))
    print()